### CArga de bases


In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()



In [2]:
query = f"""
    SELECT * FROM DANTALION.[dbo].Base_Maestra_Efectiva_Vigente
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [2]:
# fecha_mes_base='2026-07-01'

# fecha_envio_base='2026-07-01'
filename='MC_20260801_EFECTIVO_23.txt'

df_base=cargar_archivo_csv(spark,filename,'|',False)


In [8]:
df_base.show(10,truncate=False)

+----------+----------+------+--------+----------------------------------+------------------------------+----+----+----+--------------+----+----+--------------+----+----+--------+----+-----------------+----------+-------------+------+----+------+----+----+----+----+---------------------------------+----+----------+---------+---------------+---------------+---------+---------+----+----+----+----+----+----+----+----+----+----+-------------------------+----------+-------------------------------------------+---------+-------------------------+------+-----------+----+----------+----+-------+--------+--------+----+----+----+----+----+----+----+-------------+-----------------------------+
|_c0       |_c1       |_c2   |_c3     |_c4                               |_c5                           |_c6 |_c7 |_c8 |_c9           |_c10|_c11|_c12          |_c13|_c14|_c15    |_c16|_c17             |_c18      |_c19         |_c20  |_c21|_c22  |_c23|_c24|_c25|_c26|_c27                             |_c28|_c29

In [ ]:
df_base.select('_c66').distinct().show(5,truncate=False)

In [4]:
exprs = [
    F.count(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c).cast("string")) != "") &
            (F.upper(F.trim(F.col(c).cast("string"))) != "NULL"),
            c
        )
    ).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

df_base=df_base.drop('_c28','_c29','_c0','_c1','_c2')

df_base=df_base.withColumnRenamed('_c3','DNI')
df_base=df_base.withColumnRenamed('_c4','CLIENTE')
df_base=df_base.withColumnRenamed('_c5','ASIGNACION')
df_base=df_base.withColumnRenamed('_c9','ZONA')
df_base=df_base.withColumnRenamed('_c23','PERFIL')
df_base=df_base.withColumnRenamed('_c24','SEGMENTO')
df_base=df_base.withColumnRenamed('_c66','TIPOINGRESO')
df_base=df_base.withColumnRenamed('_c65','SITUACIONLABORAL')
df_base=df_base.withColumnRenamed('_c51','tipocliente')
df_base=df_base.withColumnRenamed('_c53','FECULTASIGNACION')
df_base=df_base.withColumnRenamed('_c55','LINEA_FT')
df_base=df_base.withColumnRenamed('_c56','LINEA_HS_RS')
df_base=df_base.withColumnRenamed('_c57','LINEA_HS_PLUS')

df_base=df_base.withColumnRenamed('_c45','empresa1')
df_base=df_base.withColumnRenamed('_c47','empresa2')
df_base=df_base.withColumnRenamed('_c49','empresa3')

df_base=df_base.withColumnRenamed('_c30','CELULAR1')
df_base=df_base.withColumnRenamed('_c31','CELULAR2')
df_base=df_base.withColumnRenamed('_c32','CELULAR3')
df_base=df_base.withColumnRenamed('_c33','CELULAR4')
df_base=df_base.withColumnRenamed('_c34','CELULAR5')
df_base=df_base.withColumnRenamed('_c35','CELULAR6')
df_base=df_base.withColumnRenamed('_c36','CELULAR7')
df_base=df_base.withColumnRenamed('_c37','CELULAR8')
df_base=df_base.withColumnRenamed('_c38','CELULAR9')
df_base=df_base.withColumnRenamed('_c39','CELULAR10')
df_base=df_base.withColumnRenamed('_c40','CELULAR11')
df_base=df_base.withColumnRenamed('_c41','CELULAR12')
df_base=df_base.withColumnRenamed('_c42','CELULAR13')
df_base=df_base.withColumnRenamed('_c43','CELULAR14')
df_base=df_base.withColumnRenamed('_c44','CELULAR15')

df_base=df_base.withColumnRenamed('_c11','CODIGO_AGENCIA')
df_base=df_base.withColumnRenamed('_c12','AGENCIA')
df_base=df_base.withColumnRenamed('_c15','URBANIZACION')
df_base=df_base.withColumnRenamed('_c17','DISTRITO')
df_base=df_base.withColumnRenamed('_c18','PROVINCIA')
df_base=df_base.withColumnRenamed('_c19','DEPARTAMENTO')
df_base=df_base.withColumnRenamed('_c20','TASA')

df_base=df_base.withColumnRenamed('_c22','CME_DISPONIBLE')
df_base=df_base.withColumnRenamed('_c27','NOMCOMERCIAL')
df_base=df_base.withColumnRenamed('_c25','SCORE')
df_base=df_base.withColumnRenamed('_c26','CODBASE')
df_base=df_base.withColumnRenamed('_c54','FLGCONVENIO')
df_base=df_base.withColumnRenamed('_c59','FLGSUBPROCESO_HS')
df_base=df_base.withColumnRenamed('_c46','RangoSaldo1')
df_base=df_base.withColumnRenamed('_c48','RangoSaldo2')
df_base=df_base.withColumnRenamed('_c50','RangoSaldo3')
df_base=df_base.withColumnRenamed('_c52','CODCANAL')

In [5]:
cols_cel = [f"CELULAR{i}" for i in range(1, 16)]

df_base = df_base.withColumn(
    "CELULARES_ARRAY",
    F.array(*[F.col(c).cast("string") for c in cols_cel])
)

df_base = df_base.withColumn(
    "CELULARES_VALIDOS",
    F.array_distinct(
        F.filter(
            F.transform(
                F.col("CELULARES_ARRAY"),
                lambda x: F.regexp_replace(F.trim(x), r"\D", "")
            ),
            lambda x: x.rlike(r"^9\d{8}$")
        )
    )
)

max_cel = 15

for i in range(max_cel):
    df_base = df_base.withColumn(
        f"CEL{str(i+1).zfill(2)}",
        F.expr(f"get(CELULARES_VALIDOS, {i})")
    )

df_base = df_base.drop(
    "CELULARES_ARRAY",
    "CELULARES_VALIDOS",
    *cols_cel
)

In [6]:
df_base=df_base.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base=df_base.withColumn('MES_DURACION_BASE',F.lit('08'))
df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-08-01'))
df_base=df_base.withColumn('SERVICIO',F.lit('05'))


In [7]:
cols_base = set(df_base.columns)
cols_formato = set(df_formato.columns)

solo_en_base = cols_base - cols_formato
print("Solo en df_base:", solo_en_base)
solo_en_formato = cols_formato -cols_base
print("Solo en formato:", solo_en_formato)


Solo en df_base: set()
Solo en formato: {'CELULAR8', 'FLAT3', 'REP3', 'LINEA_FT_RETANQUEO', 'FLGREPROGRAMADOCOVID19EFE', 'FLAT1', 'TELEFONO', 'LINEA_HS_PLUS_RETANQUEO', 'MARCA4', 'CELULAR4', 'CELULAR5', 'MARCA_2025', 'CELULAR13', 'TELF5', 'PERFIL_IC', 'CELULAR2', 'MARCA2', 'CELULAR10', 'TELF3', 'LINEA_HS_RS_RETANQUEO', 'FECHA', 'CELULAR6', 'TELF4', 'CELULAR3', 'DIRECCION', 'FLAT2', 'REP2', 'MICROZONA', 'MARCA5', 'RETIRO', 'PLAZA', 'TELF1', 'CELULAR14', 'LINEA_ACOTADA', 'MARCA', 'CELULAR7', 'Retail', 'TELF2', 'REP1', 'TELF6', 'CELULAR9', 'REP4', 'TELF7', 'CELULAR15', 'LINEA_FULL', 'FECVCTOPROXCUOTACPRC19EFE', 'CELULAR12', 'DEVUELTO', 'TELF8', 'CELULAR11', 'CANAL', 'MARCA3', 'LINEA_FULL_RETANQUEO', 'CELULAR1'}


In [8]:
print(df_base.count())
print(df_base.dropDuplicates(['DNI']).count())

155709
155709


In [10]:
append_table_SQL(spark,df_base,'Base_Maestra_Efectiva',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [9]:
df_base.show(5)

+--------+--------------------+--------------------+-------+--------------+-------------+------------+------------+----------+------------+------+--------------+------+--------+-----+-------+--------------------+--------------------+-----------+--------------------+-----------+--------------------+-----------+-----------+--------+----------------+-----------+--------+-----------+-------------+----------------+----------------+--------------------+---------+---------+---------+---------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------------+-----------------+-----------+--------+
|     DNI|             CLIENTE|          ASIGNACION|   ZONA|CODIGO_AGENCIA|      AGENCIA|URBANIZACION|    DISTRITO| PROVINCIA|DEPARTAMENTO|  TASA|CME_DISPONIBLE|PERFIL|SEGMENTO|SCORE|CODBASE|        NOMCOMERCIAL|            empresa1|RangoSaldo1|            empresa2|RangoSaldo2|            empresa3|RangoSaldo3|tipocliente|CODCANAL|FECULTASIGNACION|FLGCONVENIO|LINEA_FT|LINEA_HS_RS|LIN

In [ ]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva", "SP tNumeros Consumo")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva", "SP actualizar Consumo Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva", "SP actualizar Consumo SA")

In [ ]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva_Negocios", "SP tNumeros Consumo")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar Consumo Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar Consumo SA")